# Python para finanças — Taxas de retorno simples e logarítmica

Este notebook usa exclusivamente **acoes_ex.csv**, com os preços de AMBEV,
ODONTOPREV, VIVO, PETROBRAS, BBRASIL e BOVA. BOVA será a referência para comparar
as carteiras formadas pelas cinco ações.

Execute as células de cima para baixo. Os cálculos usam a precisão original;
o arredondamento é aplicado somente na exibição dos resultados.

## Importação das bibliotecas e da base de dados

Se faltar alguma biblioteca ou aparecer o erro de `nbformat` ao exibir um gráfico,
descomente e execute a instalação abaixo no próprio notebook. Depois reinicie o kernel.

In [5]:
# %pip install --upgrade pandas numpy plotly nbformat

In [6]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import display

# Hipótese didática de anualização mantida do notebook original.
# Não representa a quantidade exata de pregões de cada ano.
PREGOES_POR_ANO = 246
BENCHMARK = 'BOVA'

In [7]:
# Funciona com o kernel iniciado na pasta semana_7 ou na raiz do projeto.
caminho_csv = "/home/mgabriel4/Documentos/GitHub/python-investimentos/bases/acoes_ex.csv"
if caminho_csv is None:
    raise FileNotFoundError(
        'acoes_ex.csv não encontrado. Inicie o kernel na raiz do projeto ou em semana_7.'
    )

dataset = pd.read_csv(caminho_csv, parse_dates=['Date'])
dataset = dataset.sort_values('Date').reset_index(drop=True)
ativos = dataset.columns.drop('Date').tolist()
acoes = [ativo for ativo in ativos if ativo != BENCHMARK]

if BENCHMARK not in ativos or not acoes:
    raise ValueError('A base precisa conter BOVA e pelo menos uma ação.')
if len(dataset) < 2 or dataset['Date'].isna().any() or dataset['Date'].duplicated().any():
    raise ValueError('A base precisa ter pelo menos duas datas válidas e sem duplicatas.')

precos = dataset.set_index('Date')[ativos].apply(pd.to_numeric, errors='raise')
if not np.isfinite(precos.to_numpy()).all() or (precos <= 0).any().any():
    raise ValueError('Os preços precisam ser positivos, finitos e sem valores ausentes.')

print(f'Arquivo: {caminho_csv}')
print(f'Observações: {len(dataset)} | Ativos: {len(ativos)}')
print(f'Período: {precos.index.min():%d/%m/%Y} a {precos.index.max():%d/%m/%Y}')
display(dataset.round({ativo: 2 for ativo in ativos}))

Arquivo: /home/mgabriel4/Documentos/GitHub/python-investimentos/bases/acoes_ex.csv
Observações: 1446 | Ativos: 6
Período: 02/01/2015 a 03/11/2020


,Date,AMBEV,ODONTOPREV,VIVO,PETROBRAS,BBRASIL,BOVA
0,2015-01-02,16.02,9.37,37.82,9.00,22.65,47.26
1,2015-01-05,15.73,9.62,37.07,8.27,22.18,46.32
2,2015-01-06,16.33,9.55,36.15,8.06,22.49,46.58
3,2015-01-07,16.56,9.69,37.39,8.45,23.48,48.15
4,2015-01-08,16.54,9.66,38.91,9.02,23.56,48.51
...,...,...,...,...,...,...,...
1441,2020-10-27,13.72,14.00,43.48,19.86,32.78,95.80
1442,2020-10-28,13.37,12.97,42.61,18.64,31.14,92.43
1443,2020-10-29,12.89,12.84,42.25,19.33,30.96,92.91
1444,2020-10-30,12.18,12.61,42.67,19.00,29.80,90.66


In [8]:
dataset.shape

(1446, 7)

### Filtros por data

O acessador `.dt` permite obter o ano e o mês de cada data da coluna `Date`.

In [9]:
resultado_fevereiro = dataset.loc[
    (dataset['Date'].dt.year == 2019) & (dataset['Date'].dt.month == 2)
]
resultado_fevereiro.round({ativo: 2 for ativo in ativos})

,Date,AMBEV,ODONTOPREV,VIVO,PETROBRAS,BBRASIL,BOVA
1015,2019-02-01,18.11,16.42,42.59,30.02,52.11,94.36
1016,2019-02-04,18.45,16.57,42.80,30.36,53.10,95.03
1017,2019-02-05,18.64,16.52,42.79,30.55,54.65,94.80
1018,2019-02-06,18.09,16.40,40.49,30.01,51.32,91.27
1019,2019-02-07,18.75,16.38,41.96,29.48,51.66,91.00
1020,2019-02-08,18.22,16.25,41.72,29.27,51.73,91.70
1021,2019-02-11,18.71,16.20,42.01,28.90,50.20,90.99
1022,2019-02-12,18.67,16.24,41.88,29.94,53.29,92.45
1023,2019-02-13,18.39,16.26,42.19,30.65,52.08,92.60
1024,2019-02-14,18.59,16.47,42.30,31.29,54.74,94.34


In [10]:
resultado_2020 = dataset.loc[dataset['Date'].dt.year == 2020]
resultado_2020.round({ativo: 2 for ativo in ativos})

,Date,AMBEV,ODONTOPREV,VIVO,PETROBRAS,BBRASIL,BOVA
1241,2020-01-02,19.20,17.01,48.24,32.80,53.80,114.24
1242,2020-01-03,18.93,17.22,48.80,31.99,53.71,113.80
1243,2020-01-06,19.02,17.09,48.33,33.03,53.00,112.59
1244,2020-01-07,19.06,16.65,49.40,32.58,52.60,112.24
1245,2020-01-08,18.97,16.70,49.90,32.05,52.12,111.95
...,...,...,...,...,...,...,...
1441,2020-10-27,13.72,14.00,43.48,19.86,32.78,95.80
1442,2020-10-28,13.37,12.97,42.61,18.64,31.14,92.43
1443,2020-10-29,12.89,12.84,42.25,19.33,30.96,92.91
1444,2020-10-30,12.18,12.61,42.67,19.00,29.80,90.66


## Taxa de retorno simples no período

O retorno simples mede a variação do preço em relação ao valor inicial:

`retorno simples = preço final / preço inicial - 1`

Multiplicamos por 100 para apresentar o resultado em porcentagem. Calculamos
todos os ativos de uma vez, incluindo BOVA. Retornos sucessivos são compostos
por multiplicação dos fatores `(1 + retorno)`, e não por soma das taxas.

In [11]:
precos_iniciais = precos.iloc[0]
precos_finais = precos.iloc[-1]
retorno_simples_periodo = precos_finais / precos_iniciais - 1

resumo_periodo = pd.DataFrame({
    'Preço inicial': precos_iniciais,
    'Preço final': precos_finais,
    'Retorno simples (%)': retorno_simples_periodo * 100,
})
resumo_periodo.index.name = 'Ativo'
resumo_periodo.round(2)

,Preço inicial,Preço final,Retorno simples (%)
Ativo,,,
AMBEV,16.02,12.67,-20.91
ODONTOPREV,9.37,12.51,33.51
VIVO,37.82,42.78,13.11
PETROBRAS,9.00,19.69,118.78
BBRASIL,22.65,30.12,32.98
BOVA,47.26,92.27,95.24


### Retorno simples diário e o uso de shift

`shift(1)` desloca os preços em uma linha, permitindo comparar cada observação
com a anterior. O primeiro retorno fica ausente (`NaN`), pois não há preço anterior.
As taxas ficam em formato decimal nos cálculos: `0.01` corresponde a `1%`.

In [12]:
exemplo_ambev = pd.DataFrame({
    'Preço AMBEV': precos['AMBEV'],
    'Preço anterior AMBEV': precos['AMBEV'].shift(1),
})
exemplo_ambev.head().round(2)

,Preço AMBEV,Preço anterior AMBEV
Date,,
2015-01-02,16.02,NaN
2015-01-05,15.73,16.02
2015-01-06,16.33,15.73
2015-01-07,16.56,16.33
2015-01-08,16.54,16.56


In [13]:
retornos_simples = precos / precos.shift(1) - 1
retornos_simples.head().round(6)

,AMBEV,ODONTOPREV,VIVO,PETROBRAS,BBRASIL,BOVA
Date,,,,,,
2015-01-02,NaN,NaN,NaN,NaN,NaN,NaN
2015-01-05,-0.018102,0.026681,-0.019831,-0.081111,-0.020751,-0.019890
2015-01-06,0.038144,-0.007276,-0.024818,-0.025393,0.013977,0.005613
2015-01-07,0.014084,0.014660,0.034301,0.048387,0.044020,0.033705
2015-01-08,-0.001208,-0.003096,0.040653,0.067456,0.003407,0.007477


In [14]:
figura = px.line(
    retornos_simples * 100,
    title='Retornos simples diários — ações e BOVA',
    labels={'Date': 'Data', 'value': 'Retorno diário (%)', 'variable': 'Ativo'},
)
figura.show()

In [15]:
medias_simples_diarias = retornos_simples.mean()
(medias_simples_diarias * 100).to_frame('Média diária simples (%)').round(4)

,Média diária simples (%)
AMBEV,-0.0012
ODONTOPREV,0.0400
VIVO,0.0293
PETROBRAS,0.1124
BBRASIL,0.0649
BOVA,0.0614


### Estimativa de retorno médio anual

Mantemos o cálculo didático do notebook original: média aritmética diária
multiplicada por **246 pregões**. Essa estimativa não é o retorno acumulado
observado nem inclui capitalização. A quantidade de pregões é uma hipótese
ajustável em `PREGOES_POR_ANO`.

In [16]:
display(precos.head(PREGOES_POR_ANO).round(2))
display(precos.tail(PREGOES_POR_ANO).round(2))

,AMBEV,ODONTOPREV,VIVO,PETROBRAS,BBRASIL,BOVA
Date,,,,,,
2015-01-02,16.02,9.37,37.82,9.00,22.65,47.26
2015-01-05,15.73,9.62,37.07,8.27,22.18,46.32
2015-01-06,16.33,9.55,36.15,8.06,22.49,46.58
2015-01-07,16.56,9.69,37.39,8.45,23.48,48.15
2015-01-08,16.54,9.66,38.91,9.02,23.56,48.51
...,...,...,...,...,...,...
2015-12-22,17.95,9.57,31.38,8.54,15.00,42.23
2015-12-23,18.20,9.59,32.40,8.88,14.85,42.75
2015-12-28,18.14,9.58,30.80,8.61,14.98,42.50


,AMBEV,ODONTOPREV,VIVO,PETROBRAS,BBRASIL,BOVA
Date,,,,,,
2019-10-29,17.58,15.26,44.55,32.24,48.66,103.50
2019-10-30,17.56,15.46,44.55,32.63,49.42,104.29
2019-10-31,17.38,14.79,44.50,32.70,48.15,103.23
2019-11-01,17.33,15.18,44.26,32.74,47.96,104.27
2019-11-04,17.57,15.03,45.20,32.97,47.69,104.63
...,...,...,...,...,...,...
2020-10-27,13.72,14.00,43.48,19.86,32.78,95.80
2020-10-28,13.37,12.97,42.61,18.64,31.14,92.43
2020-10-29,12.89,12.84,42.25,19.33,30.96,92.91


In [17]:
retornos_simples_anuais = medias_simples_diarias * PREGOES_POR_ANO
(retornos_simples_anuais * 100).to_frame('Estimativa anual simples (%)').round(2)

,Estimativa anual simples (%)
AMBEV,-0.31
ODONTOPREV,9.84
VIVO,7.20
PETROBRAS,27.64
BBRASIL,15.97
BOVA,15.11


## Taxa de retorno logarítmica

`retorno logarítmico = ln(preço final / preço inicial)`

Os retornos logarítmicos podem ser somados ao longo do tempo para um mesmo ativo.
Eles são próximos dos retornos simples quando as variações são pequenas, mas
representam outra medida: para converter de logarítmico para simples, usamos
`exp(retorno logarítmico) - 1`. O logaritmo exige preços positivos.

In [18]:
retorno_log_periodo = np.log(precos_finais / precos_iniciais)
resumo_periodo['Retorno logarítmico (%)'] = retorno_log_periodo * 100
resumo_periodo.round(2)

,Preço inicial,Preço final,Retorno simples (%),Retorno logarítmico (%)
Ativo,,,,
AMBEV,16.02,12.67,-20.91,-23.46
ODONTOPREV,9.37,12.51,33.51,28.90
VIVO,37.82,42.78,13.11,12.32
PETROBRAS,9.00,19.69,118.78,78.29
BBRASIL,22.65,30.12,32.98,28.50
BOVA,47.26,92.27,95.24,66.91


In [19]:
retornos_log = np.log(precos / precos.shift(1))
retornos_log.head().round(6)

,AMBEV,ODONTOPREV,VIVO,PETROBRAS,BBRASIL,BOVA
Date,,,,,,
2015-01-02,NaN,NaN,NaN,NaN,NaN,NaN
2015-01-05,-0.018268,0.026331,-0.020030,-0.084590,-0.020969,-0.020090
2015-01-06,0.037434,-0.007303,-0.025131,-0.025721,0.013880,0.005597
2015-01-07,0.013986,0.014553,0.033726,0.047253,0.043078,0.033150
2015-01-08,-0.001208,-0.003101,0.039848,0.065278,0.003401,0.007449


In [20]:
figura = px.line(
    retornos_log * 100,
    title='Retornos logarítmicos diários — ações e BOVA',
    labels={'Date': 'Data', 'value': 'Retorno logarítmico diário (%)', 'variable': 'Ativo'},
)
figura.show()

In [21]:
medias_log_diarias = retornos_log.mean()
retornos_log_anuais = medias_log_diarias * PREGOES_POR_ANO

resumo_medias = pd.DataFrame({
    'Média diária simples (%)': medias_simples_diarias * 100,
    'Média diária logarítmica (%)': medias_log_diarias * 100,
    'Estimativa anual simples (%)': retornos_simples_anuais * 100,
    'Média logarítmica anualizada (%)': retornos_log_anuais * 100,
    'Equivalente simples da média log anualizada (%)': np.expm1(retornos_log_anuais) * 100,
})
resumo_medias.round(4)

,Média diária simples (%),Média diária logarítmica (%),Estimativa anual simples (%),Média logarítmica anualizada (%),Equivalente simples da média log anualizada (%)
AMBEV,-0.0012,-0.0162,-0.3055,-3.9939,-3.9152
ODONTOPREV,0.0400,0.0200,9.8360,4.9203,5.0433
VIVO,0.0293,0.0085,7.1963,2.0979,2.1201
PETROBRAS,0.1124,0.0542,27.6396,13.3280,14.2570
BBRASIL,0.0649,0.0197,15.9670,4.8524,4.9721
BOVA,0.0614,0.0463,15.1077,11.3901,12.0642


## Retorno de carteiras de ações

Normalizamos cada série dividindo seus preços pelo primeiro preço. Todas começam
em 1, permitindo comparar a evolução relativa dos ativos.

As carteiras usam apenas as cinco ações. BOVA fica fora dos pesos e serve como
referência. Associamos cada peso ao nome da ação para evitar depender da ordem
das colunas do CSV.

In [22]:
precos_normalizados = precos / precos_iniciais
precos_normalizados.head().round(4)

,AMBEV,ODONTOPREV,VIVO,PETROBRAS,BBRASIL,BOVA
Date,,,,,,
2015-01-02,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000
2015-01-05,0.9819,1.0267,0.9802,0.9189,0.9792,0.9801
2015-01-06,1.0194,1.0192,0.9558,0.8956,0.9929,0.9856
2015-01-07,1.0337,1.0342,0.9886,0.9389,1.0366,1.0188
2015-01-08,1.0325,1.0309,1.0288,1.0022,1.0402,1.0264


In [23]:
figura = px.line(
    precos_normalizados,
    title='Histórico dos preços normalizados — base inicial = 1',
    labels={'Date': 'Data', 'value': 'Preço normalizado', 'variable': 'Ativo'},
)
figura.show()

### Pesos das carteiras

Carteira 1: divisão igual entre as cinco ações (20% cada).

Carteira 2: AMBEV 10%, ODONTOPREV 20%, VIVO 20%, PETROBRAS 40% e BBRASIL 10%.
São os pesos do segundo exemplo original, aplicados aos ativos da nova base.

In [24]:
pesos_carteira1 = pd.Series(1 / len(acoes), index=acoes, name='Carteira 1')
pesos_carteira2 = pd.Series({
    'AMBEV': 0.10,
    'ODONTOPREV': 0.20,
    'VIVO': 0.20,
    'PETROBRAS': 0.40,
    'BBRASIL': 0.10,
}, name='Carteira 2')

for pesos in (pesos_carteira1, pesos_carteira2):
    if set(pesos.index) != set(acoes):
        raise ValueError(f'Os pesos de {pesos.name} precisam cobrir todas as ações.')
    if pesos.isna().any() or (pesos < 0).any() or not np.isclose(pesos.sum(), 1):
        raise ValueError(f'Os pesos de {pesos.name} precisam ser válidos e somar 1.')

pesos_carteira2 = pesos_carteira2.reindex(acoes)
quadro_pesos = pd.concat([pesos_carteira1, pesos_carteira2], axis=1)
display((quadro_pesos * 100).round(2).rename_axis('Peso por ativo (%)'))
display(quadro_pesos.sum().to_frame('Soma dos pesos'))

,Carteira 1,Carteira 2
Peso por ativo (%),,
AMBEV,20.0,10.0
ODONTOPREV,20.0,20.0
VIVO,20.0,20.0
PETROBRAS,20.0,40.0
BBRASIL,20.0,10.0


,Soma dos pesos
Carteira 1,1.0
Carteira 2,1.0


### Retorno diário e estimativa anual das carteiras

A soma ponderada dos retornos diários supõe **rebalanceamento a cada observação**
para manter os pesos definidos. A estimativa anual usa a média desses retornos
multiplicada por 246. Não consideramos custos, impostos ou proventos separados
dos preços fornecidos no CSV.

In [25]:
retornos_carteiras = pd.DataFrame({
    'Carteira 1': retornos_simples[acoes].dot(pesos_carteira1),
    'Carteira 2': retornos_simples[acoes].dot(pesos_carteira2),
    BENCHMARK: retornos_simples[BENCHMARK],
})
retornos_carteiras.head().round(6)

,Carteira 1,Carteira 2,BOVA
Date,,,
2015-01-02,NaN,NaN,NaN
2015-01-05,-0.022623,-0.034960,-0.019890
2015-01-06,-0.001073,-0.011364,0.005613
2015-01-07,0.031090,0.034957,0.033705
2015-01-08,0.021442,0.034714,0.007477


In [26]:
estimativas_anuais_carteiras = retornos_carteiras.mean() * PREGOES_POR_ANO
(estimativas_anuais_carteiras * 100).to_frame('Estimativa anual simples (%)').round(2)

,Estimativa anual simples (%)
Carteira 1,12.07
Carteira 2,16.03
BOVA,15.11


### Carteiras mantidas sem rebalanceamento x BOVA

Para reproduzir a comparação do notebook original, calculamos a soma ponderada
dos **preços normalizados**. Isso representa comprar com os pesos iniciais e
manter a quantidade de cada ativo, sem rebalancear. Os pesos mudam com os preços,
por isso o resultado pode diferir da carteira rebalanceada diariamente.

In [27]:
evolucao_carteiras = pd.DataFrame({
    'Carteira 1': precos_normalizados[acoes].dot(pesos_carteira1),
    'Carteira 2': precos_normalizados[acoes].dot(pesos_carteira2),
    BENCHMARK: precos_normalizados[BENCHMARK],
})
comparativo_com_acoes = precos_normalizados.join(
    evolucao_carteiras[['Carteira 1', 'Carteira 2']]
)
comparativo_com_acoes.head().round(4)

,AMBEV,ODONTOPREV,VIVO,PETROBRAS,BBRASIL,BOVA,Carteira 1,Carteira 2
Date,,,,,,,,
2015-01-02,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000
2015-01-05,0.9819,1.0267,0.9802,0.9189,0.9792,0.9801,0.9774,0.9650
2015-01-06,1.0194,1.0192,0.9558,0.8956,0.9929,0.9856,0.9766,0.9545
2015-01-07,1.0337,1.0342,0.9886,0.9389,1.0366,1.0188,1.0064,0.9871
2015-01-08,1.0325,1.0309,1.0288,1.0022,1.0402,1.0264,1.0269,1.0201


In [28]:
figura = px.line(
    comparativo_com_acoes,
    title='Ações, carteiras sem rebalanceamento e BOVA',
    labels={'Date': 'Data', 'value': 'Valor normalizado (base = 1)', 'variable': 'Série'},
)
figura.show()

In [29]:
figura = px.line(
    evolucao_carteiras,
    title='Comparativo das carteiras sem rebalanceamento x BOVA',
    labels={'Date': 'Data', 'value': 'Valor normalizado (base = 1)', 'variable': 'Série'},
)
figura.show()

### Retorno acumulado observado

O retorno acumulado das carteiras sem rebalanceamento vem do valor final
normalizado. Para as carteiras rebalanceadas, compomos os retornos diários
multiplicando os fatores `(1 + retorno)`. A primeira linha recebe fator 1
somente para iniciar o gráfico; ela não entra na média diária.

In [30]:
evolucao_rebalanceada = (1 + retornos_carteiras.fillna(0)).cumprod()

resumo_carteiras = pd.DataFrame({
    'Estimativa anual simples com rebalanceamento (%)': estimativas_anuais_carteiras * 100,
    'Retorno acumulado sem rebalanceamento (%)': (evolucao_carteiras.iloc[-1] - 1) * 100,
    'Retorno acumulado com rebalanceamento (%)': (evolucao_rebalanceada.iloc[-1] - 1) * 100,
})
resumo_carteiras.round(2)

,Estimativa anual simples com rebalanceamento (%),Retorno acumulado sem rebalanceamento (%),Retorno acumulado com rebalanceamento (%)
Carteira 1,12.07,35.49,63.01
Carteira 2,16.03,58.04,91.13
BOVA,15.11,95.24,95.24


## Exercício com a nova base

Toda a análise acima já utiliza `acoes_ex.csv`. Para experimentar outra divisão
de investimento, altere os pesos abaixo mantendo as cinco ações e a soma igual a 1.
O exemplo começa com 20% em cada ação e mantém as quantidades sem rebalanceamento.

In [31]:
pesos_exercicio = pd.Series({
    'AMBEV': 0.20,
    'ODONTOPREV': 0.20,
    'VIVO': 0.20,
    'PETROBRAS': 0.20,
    'BBRASIL': 0.20,
})
if set(pesos_exercicio.index) != set(acoes):
    raise ValueError('Defina um peso para cada ação da base.')
if pesos_exercicio.isna().any() or (pesos_exercicio < 0).any() or not np.isclose(pesos_exercicio.sum(), 1):
    raise ValueError('Os pesos do exercício precisam ser válidos e somar 1.')
pesos_exercicio = pesos_exercicio.reindex(acoes)

comparativo_exercicio = pd.DataFrame({
    'Carteira do exercício': precos_normalizados[acoes].dot(pesos_exercicio),
    BENCHMARK: precos_normalizados[BENCHMARK],
})
(comparativo_exercicio.iloc[-1] - 1).mul(100).to_frame('Retorno acumulado (%)').round(2)

,Retorno acumulado (%)
Carteira do exercício,35.49
BOVA,95.24


In [32]:
figura = px.line(
    comparativo_exercicio,
    title='Carteira do exercício sem rebalanceamento x BOVA',
    labels={'Date': 'Data', 'value': 'Valor normalizado (base = 1)', 'variable': 'Série'},
)
figura.show()